In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Convert timestamp to timezone-aware datetimes (UTC->Asia/Shanghai) and keep as new column 'dt'
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper to compute group count and global P95 on 'value'
def group_count_p95(df, comp_col, kpi_col):
    # Ensure 'value' is numeric
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    g = df.groupby([comp_col, kpi_col])['value'].agg(count='count', p95_value=lambda x: x.quantile(0.95)).reset_index()
    g = g.sort_values('count', ascending=False)
    # rename component and KPI columns to required names
    g = g.rename(columns={comp_col: 'component', kpi_col: 'KPI name'})
    return g

# Metric summary (cmdb_id, kpi_name)
metric_summary = group_count_p95(metric_df, 'cmdb_id', 'kpi_name')
metric_top20 = metric_summary.head(20)

# Trace summary (cmdb_id, trace_name)
trace_summary = group_count_p95(trace_df, 'cmdb_id', 'trace_name')
trace_top20 = trace_summary.head(20)

# Log summary (cmdb_id, log_name)
log_summary = group_count_p95(log_df, 'cmdb_id', 'log_name')
log_top20 = log_summary.head(20)

# Error logs: (a) counts of error log rows by cmdb_id (top 20)
error_counts = error_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)
error_counts_top20 = error_counts.head(20)

# (b) top 20 raw rows with selected columns; ensure dt (tz-aware) shown
error_raw_top20 = error_df[['dt', 'cmdb_id', 'message']].head(20)

# Display compact summaries (variables as output)
metric_top20, trace_top20, log_top20, error_counts_top20, error_raw_top20

```
Out[1]:
```
```python
# Summarize the previously computed top groups and error logs as a compact English summary string.
# Reuse existing DataFrame variables from the session: metric_top20, trace_top20, log_top20, error_counts_top20, error_raw_top20

def fmt_row(r):
    p95 = r.get('p95_value', None)
    if pd.isna(p95):
        p95s = "p95=NaN"
    else:
        # use compact formatting
        p95s = f"p95={p95:.6g}"
    return f"{r.get('component')} | {r.get('KPI name')} | count={int(r.get('count'))} | {p95s}"

lines = []
lines.append("Summary of top telemetry groups (based on full available series):")

# Metric summary (show top 5)
lines.append("\nMetrics (top 5 groups by count):")
for _, row in metric_top20.head(5).iterrows():
    lines.append("  - " + fmt_row(row))

# Trace summary (top 5)
lines.append("\nTraces (top 5 groups by count):")
for _, row in trace_top20.head(5).iterrows():
    lines.append("  - " + fmt_row(row))

# Log summary (top 5)
lines.append("\nLogs (top 5 groups by count):")
for _, row in log_top20.head(5).iterrows():
    lines.append("  - " + fmt_row(row))

# Error logs summary
lines.append("\nError logs:")
if error_counts_top20.shape[0] == 0:
    lines.append("  - No error log rows found in error_logs.csv (empty result).")
else:
    lines.append("  - Counts by component (top entries):")
    for _, r in error_counts_top20.head(5).iterrows():
        lines.append(f"    * {r['cmdb_id']}: count={int(r['count'])}")
    # raw rows
    if error_raw_top20.shape[0] == 0:
        lines.append("  - No raw error log rows available to display.")
    else:
        lines.append("  - Top raw error log rows (timestamp, component, message) shown separately.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     component                                        KPI name  count     p95_value
6         IG01        JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.350000e+01
3         IG01     JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.792518e+07
2         IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  8.262197e+08
0         IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09
1663  apache02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00
15        IG01                          OSLinux-CPU_CPU_CPUWio     31  6.113635e-01
14        IG01                     OSLinux-CPU_CPU_CPUUserTime     31  2.538880e+01
13        IG01                      OSLinux-CPU_CPU_CPUSysTime     31  5.343500e-01
824    Redis01                     OSLinux-CPU_CPU_CPUidleutil     31  7.284572e+01
820    Redis01                         OSLinux-CPU_CPU_CPULoad     31  1.465000e+00
821    Redis01                      OSLinux-CPU_CPU_CPUSysTime     31  2.633900e+00
822    Redis01                     OSLinux-CPU_CPU_CPUUserTime     31  2.523094e+01
823    Redis01                          OSLinux-CPU_CPU_CPUWio     31  1.186500e-01
876    Redis01    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  1.000000e+00
104       IG02                     OSLinux-CPU_CPU_CPUUserTime     31  2.547695e+01
105       IG02                          OSLinux-CPU_CPU_CPUWio     31  4.002476e+00
106       IG02                     OSLinux-CPU_CPU_CPUidleutil     31  7.380343e+01
96        IG02        JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.500000e+01
102       IG02                         OSLinux-CPU_CPU_CPULoad     31  1.225000e+00
67        IG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00,     component                       KPI name  count    p95_value
263  dockerB2        trace.to_MG02.row_count     30   217.150000
262  dockerB2     trace.to_MG02.duration_p95     30    22.131042
261  dockerB2    trace.to_MG02.duration_mean     30    10.900926
260  dockerB2        trace.to_MG01.row_count     30   223.950000
3        IG01       trace.self.duration_mean     30     3.178588
221  dockerA2           trace.self.row_count     30  2916.100000
222  dockerA2    trace.to_MG01.duration_mean     30    10.561496
223  dockerA2     trace.to_MG01.duration_p95     30    19.060112
224  dockerA2        trace.to_MG01.row_count     30    62.550000
225  dockerA2    trace.to_MG02.duration_mean     30    11.541783
226  dockerA2     trace.to_MG02.duration_p95     30    19.456600
227  dockerA2        trace.to_MG02.row_count     30    65.650000
228  dockerB1  trace.from_MG01.duration_mean     30     6.272658
213  dockerA2  trace.from_MG02.duration_mean     30     2.811166
214  dockerA2   trace.from_MG02.duration_p95     30    14.620012
215  dockerA2      trace.from_MG02.row_count     30   214.800000
219  dockerA2       trace.self.duration_mean     30     0.375762
220  dockerA2        trace.self.duration_p95     30     1.082782
205  dockerA1     trace.to_MG01.duration_p95     30    21.238150
206  dockerA1        trace.to_MG01.row_count     30    63.100000,    component          KPI name  count  p95_value
9   Tomcat02     log.row_count     31   508.0000
7   Tomcat02   log.error_count     31     0.0000
6   Tomcat01     log.row_count     31   511.5000
4   Tomcat01   log.error_count     31     0.0000
18  apache02   log.error_count     31     0.0000
17  apache01     log.row_count     31  1018.0000
19  apache02     log.row_count     31  1047.5000
16  apache01   log.error_count     31     0.0000
10  Tomcat03   log.error_count     31     0.0000
15  Tomcat04     log.row_count     31   518.5000
13  Tomcat04   log.error_count     31     0.0000
12  Tomcat03     log.row_count     31   529.0000
8   Tomcat02  log.gc_pause_max     30     0.1000
0    Mysql01   log.error_count     25     0.0000
1    Mysql01     log.row_count     25    12.4000
11  Tomcat03  log.gc_pause_max     23     0.0990
5   Tomcat01  log.gc_pause_max     22     0.0995
14  Tomcat04  log.gc_pause_max     22     0.1190
3    Mysql02     log.row_count      3     1.9000
2    Mysql02   log.error_count      3     0.0000, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [dt, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames if present; otherwise load
try:
    metric_df
    trace_df
    log_df
    error_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')

# Ensure 'dt' tz-aware column exists for all files (UTC -> Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_df):
    if 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure 'value' numeric where applicable
metric_df['value'] = pd.to_numeric(metric_df.get('value', pd.Series()), errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df.get('value', pd.Series()), errors='coerce')
log_df['value'] = pd.to_numeric(log_df.get('value', pd.Series()), errors='coerce')

# Helper to compute global P95 per group (full series BEFORE filtering)
def compute_p95(df, comp_col, kpi_col):
    g = df.groupby([comp_col, kpi_col])['value'].agg(count='count', p95_value=lambda x: x.quantile(0.95)).reset_index()
    g = g.rename(columns={comp_col: 'component', kpi_col: 'KPI name'})
    return g

metric_p95 = compute_p95(metric_df, 'cmdb_id', 'kpi_name')
trace_p95  = compute_p95(trace_df,  'cmdb_id', 'trace_name')
log_p95    = compute_p95(log_df,    'cmdb_id', 'log_name')

# Define time window in UTC+8
start_dt = tz.localize(pd.Timestamp('2021-03-10 15:30:00'))
end_dt   = tz.localize(pd.Timestamp('2021-03-10 16:00:00'))

# Filter original files to the time window (use dt column)
metric_win = metric_df[(metric_df['dt'] >= start_dt) & (metric_df['dt'] <= end_dt)].copy()
trace_win  = trace_df[(trace_df['dt'] >= start_dt) & (trace_df['dt'] <= end_dt)].copy()
log_win    = log_df[(log_df['dt'] >= start_dt) & (log_df['dt'] <= end_dt)].copy()
error_win  = error_df[(error_df['dt'] >= start_dt) & (error_df['dt'] <= end_dt)].copy()

# Function to mark anomalies by merging global p95 and checking value >= p95
def mark_anomalies(win_df, p95_df, comp_col, kpi_col):
    merged = win_df.merge(p95_df.rename(columns={'component': comp_col, 'KPI name': kpi_col}),
                          on=[comp_col, kpi_col], how='left')
    # mark anomaly where value >= p95_value (NaN thresholds -> False)
    merged['is_anomaly'] = merged['value'].notna() & merged['p95_value'].notna() & (merged['value'] >= merged['p95_value'])
    return merged

metric_annot = mark_anomalies(metric_win, metric_p95.rename(columns={'component':'cmdb_id','KPI name':'kpi_name'}), 'cmdb_id', 'kpi_name')
trace_annot  = mark_anomalies(trace_win,  trace_p95.rename(columns={'component':'cmdb_id','KPI name':'trace_name'}), 'cmdb_id', 'trace_name')
log_annot    = mark_anomalies(log_win,    log_p95.rename(columns={'component':'cmdb_id','KPI name':'log_name'}), 'cmdb_id', 'log_name')

# Aggregation helpers
def per_component_summary(annot_df, comp_col):
    s = annot_df[annot_df['is_anomaly']].groupby(comp_col).agg(
        total_anomalies=('is_anomaly','sum'),
        earliest_anomaly_timestamp=('dt','min')
    ).reset_index().rename(columns={comp_col:'component'})
    s = s.sort_values('total_anomalies', ascending=False).head(20)
    return s

def per_compkpi_summary(annot_df, comp_col, kpi_col):
    s = annot_df[annot_df['is_anomaly']].groupby([comp_col, kpi_col]).agg(
        anomaly_count=('is_anomaly','sum'),
        earliest_anomaly_timestamp=('dt','min')
    ).reset_index().rename(columns={comp_col:'component', kpi_col:'KPI name'})
    s = s.sort_values('anomaly_count', ascending=False).head(20)
    return s

def top5_raw_anomalies_concat(annot_df, comp_col, kpi_col):
    compkpi_summary_full = annot_df[annot_df['is_anomaly']].groupby([comp_col, kpi_col]).size().reset_index(name='count')
    top5 = compkpi_summary_full.sort_values('count', ascending=False).head(5)
    # For each top pair, get up to 20 raw anomaly rows
    parts = []
    for _, r in top5.iterrows():
        comp = r[comp_col]; kpi = r[kpi_col]
        df = annot_df[(annot_df[comp_col]==comp) & (annot_df[kpi_col]==kpi) & (annot_df['is_anomaly'])].copy()
        df = df.sort_values('dt').head(20)
        # standardize columns: timestamp, cmdb_id, KPI name, value, group_key
        df_out = df[['dt', comp_col, kpi_col, 'value']].rename(columns={'dt':'timestamp', comp_col:'cmdb_id', kpi_col:'KPI name'})
        df_out['group_key'] = f"{comp} | {kpi}"
        parts.append(df_out)
    if parts:
        return pd.concat(parts, ignore_index=True)
    else:
        return pd.DataFrame(columns=['timestamp','cmdb_id','KPI name','value','group_key'])

# Produce outputs for metric
metric_comp_anom = per_component_summary(metric_annot, 'cmdb_id')
metric_compkpi_anom = per_compkpi_summary(metric_annot, 'cmdb_id', 'kpi_name')
metric_top5_raw = top5_raw_anomalies_concat(metric_annot, 'cmdb_id', 'kpi_name')

# Produce outputs for trace
trace_comp_anom = per_component_summary(trace_annot, 'cmdb_id')
trace_compkpi_anom = per_compkpi_summary(trace_annot, 'cmdb_id', 'trace_name')
trace_top5_raw = top5_raw_anomalies_concat(trace_annot, 'cmdb_id', 'trace_name')

# Produce outputs for log
log_comp_anom = per_component_summary(log_annot, 'cmdb_id')
log_compkpi_anom = per_compkpi_summary(log_annot, 'cmdb_id', 'log_name')
log_top5_raw = top5_raw_anomalies_concat(log_annot, 'cmdb_id', 'log_name')

# Error logs within window: top 20 rows and counts per component top20
error_window_rows_top20 = error_win[['dt','cmdb_id','message']].sort_values('dt').head(20)
error_counts_top20 = error_win.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).head(20)

# Compact display: top summaries and samples (adhere to top-20 limits)
metric_comp_anom.head(20), metric_compkpi_anom.head(20), metric_top5_raw.head(100), \
trace_comp_anom.head(20), trace_compkpi_anom.head(20), trace_top5_raw.head(100), \
log_comp_anom.head(20), log_compkpi_anom.head(20), log_top5_raw.head(100), \
error_window_rows_top20, error_counts_top20

```
Out[2]:
```
```python
# Compose a compact plain-English summary from the previously computed results.
# Reuse session variables: metric_comp_anom, metric_compkpi_anom, trace_comp_anom, trace_compkpi_anom,
# log_comp_anom, log_compkpi_anom, metric_top5_raw, trace_top5_raw, log_top5_raw, error_window_rows_top20, error_counts_top20

def top_list(df, n=5, name_col='component', count_col=None):
    rows = []
    for _, r in df.head(n).iterrows():
        cnt = int(r[count_col]) if count_col and count_col in r.index else None
        ts = r.get('earliest_anomaly_timestamp')
        ts_str = pd.NaT if ts is pd.NaT else (str(ts) if pd.notna(ts) else '')
        if cnt is not None:
            rows.append(f"{r[name_col]} (count={cnt}, earliest={ts_str})")
        else:
            rows.append(f"{r[name_col]} (earliest={ts_str})")
    return "; ".join(rows) if rows else "none"

import pandas as pd

summary_lines = []
summary_lines.append("Summary of anomaly detection (window: 2021-03-10 15:30–16:00 UTC+8).")
summary_lines.append("")

# Metrics
summary_lines.append("Metrics (top components by total anomalies):")
if metric_comp_anom.empty:
    summary_lines.append("  - No metric anomalies found.")
else:
    summary_lines.append("  - Top components: " + top_list(metric_comp_anom, 5, 'component', 'total_anomalies'))
summary_lines.append("  - Top component-KPI pairs: " + top_list(metric_compkpi_anom, 5, 'KPI name', 'anomaly_count'))
summary_lines.append("  - Example top metric anomaly (first rows for the top pair shown separately).")

# Traces
summary_lines.append("")
summary_lines.append("Traces (top components by total anomalies):")
if trace_comp_anom.empty:
    summary_lines.append("  - No trace anomalies found.")
else:
    summary_lines.append("  - Top components: " + top_list(trace_comp_anom, 5, 'component', 'total_anomalies'))
summary_lines.append("  - Top component-KPI pairs: " + top_list(trace_compkpi_anom, 5, 'KPI name', 'anomaly_count'))
summary_lines.append("  - Example top trace anomaly rows provided for the top pairs.")

# Logs
summary_lines.append("")
summary_lines.append("Logs (top components by total anomalies):")
if log_comp_anom.empty:
    summary_lines.append("  - No log anomalies found.")
else:
    summary_lines.append("  - Top components: " + top_list(log_comp_anom, 5, 'component', 'total_anomalies'))
summary_lines.append("  - Top component-KPI pairs: " + top_list(log_compkpi_anom, 5, 'KPI name', 'anomaly_count'))
summary_lines.append("  - Many top log anomalies are 'log.error_count' with full-window occurrences starting at 15:30.")

# Error logs
summary_lines.append("")
if error_window_rows_top20.empty and error_counts_top20.empty:
    summary_lines.append("Error logs: no error_log rows found within the time window (empty).")
else:
    # show counts top entries
    if not error_counts_top20.empty:
        top_err = "; ".join([f"{r['cmdb_id']} (count={int(r['count'])})" for _, r in error_counts_top20.iterrows()])
        summary_lines.append("Error logs counts (top): " + top_err)
    if not error_window_rows_top20.empty:
        summary_lines.append(f"Error log rows (sample up to 20) start at {str(error_window_rows_top20['dt'].min())}.")

# Compact conclusion
summary_lines.append("")
summary_lines.append("Conclusion: Many anomalies are concentrated in database (Mysql01/Mysql02), Redis, Tomcat and apache components during the 15:30–16:00 window. Key repeated anomalous KPIs include JVM memory/heap maxima and log.error_count (seen across Tomcat/apache). No raw error_log entries were found in the window.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        component  total_anomalies earliest_anomaly_timestamp
4         Mysql01             2191  2021-03-10 15:30:00+08:00
5         Mysql02             2076  2021-03-10 15:30:00+08:00
7         Redis02             1340  2021-03-10 15:30:00+08:00
6         Redis01             1281  2021-03-10 15:30:00+08:00
20       Tomcat02              994  2021-03-10 15:30:00+08:00
21       Tomcat03              982  2021-03-10 15:30:00+08:00
22       Tomcat04              965  2021-03-10 15:30:00+08:00
19       Tomcat01              883  2021-03-10 15:30:00+08:00
23       apache01              669  2021-03-10 15:30:00+08:00
2            MG01              656  2021-03-10 15:30:00+08:00
3            MG02              653  2021-03-10 15:30:00+08:00
0            IG01              644  2021-03-10 15:30:00+08:00
1            IG02              630  2021-03-10 15:30:00+08:00
24       apache02              502  2021-03-10 15:30:00+08:00
8    ServiceTest1               66  2021-03-10 15:30:00+08:00
10  ServiceTest11               66  2021-03-10 15:30:00+08:00
9   ServiceTest10               66  2021-03-10 15:30:00+08:00
14   ServiceTest5               66  2021-03-10 15:30:00+08:00
11   ServiceTest2               66  2021-03-10 15:30:00+08:00
17   ServiceTest8               66  2021-03-10 15:30:00+08:00,      component                                           KPI name  anomaly_count earliest_anomaly_timestamp
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-10 15:30:00+08:00
1663  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-10 15:30:00+08:00
1589  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-10 15:30:00+08:00
1393  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31  2021-03-10 15:30:00+08:00
1407  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31  2021-03-10 15:30:00+08:00
1405  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-10 15:30:00+08:00
1404  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-10 15:30:00+08:00
1588  apache01   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount             31  2021-03-10 15:30:00+08:00
1395  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31  2021-03-10 15:30:00+08:00
1394  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31  2021-03-10 15:30:00+08:00
1288  Tomcat02     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31  2021-03-10 15:30:00+08:00
1289  Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31  2021-03-10 15:30:00+08:00
578    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-10 15:30:00+08:00
1399  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-10 15:30:00+08:00
1389  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31  2021-03-10 15:30:00+08:00
1390  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-10 15:30:00+08:00
1290  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-10 15:30:00+08:00
1291  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-10 15:30:00+08:00
1398  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31  2021-03-10 15:30:00+08:00
1292  Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-10 15:30:00+08:00,                    timestamp   cmdb_id                                           KPI name         value                                          group_key
0  2021-03-10 15:30:00+08:00      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09    IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax
1  2021-03-10 15:31:00+08:00      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09    IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax
2  2021-03-10 15:32:00+08:00      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09    IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax
3  2021-03-10 15:33:00+08:00      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09    IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax
4  2021-03-10 15:34:00+08:00      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09    IG01 | JVM-Memory_7778_JVM_Memory_HeapMemoryMax
..                       ...       ...                                                ...           ...                                                ...
95 2021-03-10 15:45:00+08:00  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"...
96 2021-03-10 15:46:00+08:00  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"...
97 2021-03-10 15:47:00+08:00  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"...
98 2021-03-10 15:48:00+08:00  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"...
99 2021-03-10 15:49:00+08:00  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  Tomcat03 | Tomcat-Threads_7441-"http-nio-8003"...

[100 rows x 5 columns],    component  total_anomalies earliest_anomaly_timestamp
3       MG02               83  2021-03-10 15:30:00+08:00
2       MG01               82  2021-03-10 15:30:00+08:00
6   Tomcat03               34  2021-03-10 15:30:00+08:00
4   Tomcat01               34  2021-03-10 15:30:00+08:00
0       IG01               33  2021-03-10 15:30:00+08:00
1       IG02               33  2021-03-10 15:30:00+08:00
5   Tomcat02               33  2021-03-10 15:30:00+08:00
7   Tomcat04               33  2021-03-10 15:30:00+08:00
8   dockerA1               33  2021-03-10 15:30:00+08:00
9   dockerA2               33  2021-03-10 15:30:00+08:00
10  dockerB1               33  2021-03-10 15:30:00+08:00
11  dockerB2               33  2021-03-10 15:30:00+08:00,     component                       KPI name  anomaly_count earliest_anomaly_timestamp
38       MG01  trace.from_Tomcat01.row_count              3  2021-03-10 15:30:00+08:00
86       MG02  trace.from_Tomcat03.row_count              3  2021-03-10 15:30:00+08:00
173  Tomcat03        trace.to_MG02.row_count              3  2021-03-10 15:30:00+08:00
134  Tomcat01        trace.to_MG01.row_count              3  2021-03-10 15:30:00+08:00
261  dockerB2    trace.to_MG02.duration_mean              2  2021-03-10 15:37:00+08:00
221  dockerA2           trace.self.row_count              2  2021-03-10 15:37:00+08:00
222  dockerA2    trace.to_MG01.duration_mean              2  2021-03-10 15:37:00+08:00
223  dockerA2     trace.to_MG01.duration_p95              2  2021-03-10 15:36:00+08:00
224  dockerA2        trace.to_MG01.row_count              2  2021-03-10 15:30:00+08:00
225  dockerA2    trace.to_MG02.duration_mean              2  2021-03-10 15:36:00+08:00
226  dockerA2     trace.to_MG02.duration_p95              2  2021-03-10 15:38:00+08:00
227  dockerA2        trace.to_MG02.row_count              2  2021-03-10 15:42:00+08:00
228  dockerB1  trace.from_MG01.duration_mean              2  2021-03-10 15:36:00+08:00
213  dockerA2  trace.from_MG02.duration_mean              2  2021-03-10 15:38:00+08:00
214  dockerA2   trace.from_MG02.duration_p95              2  2021-03-10 15:38:00+08:00
215  dockerA2      trace.from_MG02.row_count              2  2021-03-10 15:30:00+08:00
219  dockerA2       trace.self.duration_mean              2  2021-03-10 15:38:00+08:00
220  dockerA2        trace.self.duration_p95              2  2021-03-10 15:42:00+08:00
205  dockerA1     trace.to_MG01.duration_p95              2  2021-03-10 15:36:00+08:00
206  dockerA1        trace.to_MG01.row_count              2  2021-03-10 15:42:00+08:00,                    timestamp   cmdb_id                       KPI name       value                               group_key
0  2021-03-10 15:30:00+08:00      MG01  trace.from_Tomcat01.row_count  250.000000    MG01 | trace.from_Tomcat01.row_count
1  2021-03-10 15:32:00+08:00      MG01  trace.from_Tomcat01.row_count  259.000000    MG01 | trace.from_Tomcat01.row_count
2  2021-03-10 15:37:00+08:00      MG01  trace.from_Tomcat01.row_count  250.000000    MG01 | trace.from_Tomcat01.row_count
3  2021-03-10 15:30:00+08:00      MG02  trace.from_Tomcat03.row_count  274.000000    MG02 | trace.from_Tomcat03.row_count
4  2021-03-10 15:31:00+08:00      MG02  trace.from_Tomcat03.row_count  273.000000    MG02 | trace.from_Tomcat03.row_count
5  2021-03-10 15:32:00+08:00      MG02  trace.from_Tomcat03.row_count  273.000000    MG02 | trace.from_Tomcat03.row_count
6  2021-03-10 15:30:00+08:00  Tomcat03        trace.to_MG02.row_count  274.000000      Tomcat03 | trace.to_MG02.row_count
7  2021-03-10 15:31:00+08:00  Tomcat03        trace.to_MG02.row_count  273.000000      Tomcat03 | trace.to_MG02.row_count
8  2021-03-10 15:32:00+08:00  Tomcat03        trace.to_MG02.row_count  273.000000      Tomcat03 | trace.to_MG02.row_count
9  2021-03-10 15:30:00+08:00  Tomcat01        trace.to_MG01.row_count  250.000000      Tomcat01 | trace.to_MG01.row_count
10 2021-03-10 15:32:00+08:00  Tomcat01        trace.to_MG01.row_count  259.000000      Tomcat01 | trace.to_MG01.row_count
11 2021-03-10 15:37:00+08:00  Tomcat01        trace.to_MG01.row_count  250.000000      Tomcat01 | trace.to_MG01.row_count
12 2021-03-10 15:37:00+08:00  dockerB2    trace.to_MG02.duration_mean   11.844081  dockerB2 | trace.to_MG02.duration_mean
13 2021-03-10 15:38:00+08:00  dockerB2    trace.to_MG02.duration_mean   11.583684  dockerB2 | trace.to_MG02.duration_mean,   component  total_anomalies earliest_anomaly_timestamp
3  Tomcat02               37  2021-03-10 15:30:00+08:00
2  Tomcat01               35  2021-03-10 15:30:00+08:00
5  Tomcat04               35  2021-03-10 15:30:00+08:00
4  Tomcat03               35  2021-03-10 15:30:00+08:00
6  apache01               33  2021-03-10 15:30:00+08:00
7  apache02               33  2021-03-10 15:30:00+08:00
0   Mysql01               27  2021-03-10 15:30:00+08:00
1   Mysql02                4  2021-03-10 15:31:00+08:00,    component          KPI name  anomaly_count earliest_anomaly_timestamp
13  Tomcat04   log.error_count             31  2021-03-10 15:30:00+08:00
10  Tomcat03   log.error_count             31  2021-03-10 15:30:00+08:00
7   Tomcat02   log.error_count             31  2021-03-10 15:30:00+08:00
4   Tomcat01   log.error_count             31  2021-03-10 15:30:00+08:00
16  apache01   log.error_count             31  2021-03-10 15:30:00+08:00
18  apache02   log.error_count             31  2021-03-10 15:30:00+08:00
0    Mysql01   log.error_count             25  2021-03-10 15:30:00+08:00
8   Tomcat02  log.gc_pause_max              4  2021-03-10 15:30:00+08:00
2    Mysql02   log.error_count              3  2021-03-10 15:31:00+08:00
1    Mysql01     log.row_count              2  2021-03-10 15:30:00+08:00
11  Tomcat03  log.gc_pause_max              2  2021-03-10 15:37:00+08:00
6   Tomcat01     log.row_count              2  2021-03-10 15:30:00+08:00
15  Tomcat04     log.row_count              2  2021-03-10 15:30:00+08:00
9   Tomcat02     log.row_count              2  2021-03-10 15:30:00+08:00
5   Tomcat01  log.gc_pause_max              2  2021-03-10 15:32:00+08:00
12  Tomcat03     log.row_count              2  2021-03-10 15:30:00+08:00
17  apache01     log.row_count              2  2021-03-10 15:30:00+08:00
14  Tomcat04  log.gc_pause_max              2  2021-03-10 15:34:00+08:00
19  apache02     log.row_count              2  2021-03-10 15:31:00+08:00
3    Mysql02     log.row_count              1  2021-03-10 15:33:00+08:00,                    timestamp   cmdb_id         KPI name  value                   group_key
0  2021-03-10 15:30:00+08:00  Tomcat04  log.error_count    0.0  Tomcat04 | log.error_count
1  2021-03-10 15:31:00+08:00  Tomcat04  log.error_count    0.0  Tomcat04 | log.error_count
2  2021-03-10 15:32:00+08:00  Tomcat04  log.error_count    0.0  Tomcat04 | log.error_count
3  2021-03-10 15:33:00+08:00  Tomcat04  log.error_count    0.0  Tomcat04 | log.error_count
4  2021-03-10 15:34:00+08:00  Tomcat04  log.error_count    0.0  Tomcat04 | log.error_count
..                       ...       ...              ...    ...                         ...
95 2021-03-10 15:45:00+08:00  apache01  log.error_count    0.0  apache01 | log.error_count
96 2021-03-10 15:46:00+08:00  apache01  log.error_count    0.0  apache01 | log.error_count
97 2021-03-10 15:47:00+08:00  apache01  log.error_count    0.0  apache01 | log.error_count
98 2021-03-10 15:48:00+08:00  apache01  log.error_count    0.0  apache01 | log.error_count
99 2021-03-10 15:49:00+08:00  apache01  log.error_count    0.0  apache01 | log.error_count

[100 rows x 5 columns], Empty DataFrame
Columns: [dt, cmdb_id, message]
Index: [], Empty DataFrame
Columns: [cmdb_id, count]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously computed annotated window DataFrames if available, otherwise recreate minimal state
try:
    metric_annot, trace_annot, log_annot
    metric_p95, trace_p95, log_p95
except NameError:
    # load files and recreate p95 and annotated window (keeps consistent with prior steps)
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')
    for df in (metric_df, trace_df, log_df, error_df):
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_df['value'] = pd.to_numeric(metric_df.get('value', pd.Series()), errors='coerce')
    trace_df['value'] = pd.to_numeric(trace_df.get('value', pd.Series()), errors='coerce')
    log_df['value'] = pd.to_numeric(log_df.get('value', pd.Series()), errors='coerce')

    def compute_p95(df, comp_col, kpi_col):
        return df.groupby([comp_col, kpi_col])['value'].agg(count='count', p95_value=lambda x: x.quantile(0.95)).reset_index().rename(columns={comp_col:'component', kpi_col:'KPI name'})

    metric_p95 = compute_p95(metric_df, 'cmdb_id', 'kpi_name')
    trace_p95  = compute_p95(trace_df,  'cmdb_id', 'trace_name')
    log_p95    = compute_p95(log_df,    'cmdb_id', 'log_name')

    # filter window
    start_dt = tz.localize(pd.Timestamp('2021-03-10 15:30:00'))
    end_dt   = tz.localize(pd.Timestamp('2021-03-10 16:00:00'))
    metric_win = metric_df[(metric_df['dt'] >= start_dt) & (metric_df['dt'] <= end_dt)].copy()
    trace_win  = trace_df[(trace_df['dt'] >= start_dt) & (trace_df['dt'] <= end_dt)].copy()
    log_win    = log_df[(log_df['dt'] >= start_dt) & (log_df['dt'] <= end_dt)].copy()

    # annotate anomalies by merging p95
    metric_annot = metric_win.merge(metric_p95.rename(columns={'component':'cmdb_id','KPI name':'kpi_name'}), on=['cmdb_id','kpi_name'], how='left')
    metric_annot['is_anomaly'] = metric_annot['value'].notna() & metric_annot['p95_value'].notna() & (metric_annot['value'] >= metric_annot['p95_value'])

    trace_annot = trace_win.merge(trace_p95.rename(columns={'component':'cmdb_id','KPI name':'trace_name'}), on=['cmdb_id','trace_name'], how='left')
    trace_annot['is_anomaly'] = trace_annot['value'].notna() & trace_annot['p95_value'].notna() & (trace_annot['value'] >= trace_annot['p95_value'])

    log_annot = log_win.merge(log_p95.rename(columns={'component':'cmdb_id','KPI name':'log_name'}), on=['cmdb_id','log_name'], how='left')
    log_annot['is_anomaly'] = log_annot['value'].notna() & log_annot['p95_value'].notna() & (log_annot['value'] >= log_annot['p95_value'])

# Helper to extract runs for an annotated DataFrame
def extract_runs(annot_df, comp_col, kpi_col, kpi_label):
    # consider only anomaly rows
    anoms = annot_df[annot_df['is_anomaly']].copy()
    if anoms.empty:
        return pd.DataFrame(columns=['component','KPI name','run_start_timestamp','run_end_timestamp',
                                     'run_length','run_max_value','group_p95','relative_deviation','_group_key'])
    # sort and compute minute diffs
    anoms = anoms.sort_values([comp_col, kpi_col, 'dt'])
    # create a group key for component+kpi
    anoms['_group_key'] = anoms[comp_col].astype(str) + "||" + anoms[kpi_col].astype(str)
    # compute run id per group: new run when minute gap != 1
    def assign_runs(df):
        df = df.sort_values('dt').copy()
        diff_min = df['dt'].diff().dt.total_seconds().div(60)
        # first row starts new run
        run_id = (diff_min != 1).cumsum()
        df['_run_id'] = run_id.values
        return df
    anoms = anoms.groupby('_group_key', group_keys=False).apply(assign_runs)
    # aggregate runs
    agg = anoms.groupby(['_group_key','_run_id']).agg(
        component=(comp_col,'first'),
        KPI_name=(kpi_col,'first'),
        run_start_timestamp=('dt','min'),
        run_end_timestamp=('dt','max'),
        run_length=('dt','count'),
        run_max_value=('value','max'),
        group_p95=('p95_value','first')
    ).reset_index(drop=False)
    # compute relative_deviation
    def rel_dev(row):
        gp = row['group_p95']
        rm = row['run_max_value']
        if pd.isna(gp) or gp == 0:
            return rm
        else:
            return (rm - gp) / gp
    agg['relative_deviation'] = agg.apply(rel_dev, axis=1)
    # rename columns per spec and keep internal key for joining later
    agg = agg.rename(columns={'KPI_name':'KPI name', 'run_length':'run_length_in_minutes'})
    agg['_group_key'] = agg['_group_key']
    return agg[['component','KPI name','run_start_timestamp','run_end_timestamp','run_length_in_minutes','run_max_value','group_p95','relative_deviation','_group_key']]

# Extract runs from metric, trace, log
metric_runs = extract_runs(metric_annot, 'cmdb_id', 'kpi_name', 'kpi_name')
trace_runs  = extract_runs(trace_annot,  'cmdb_id', 'trace_name', 'trace_name')
log_runs    = extract_runs(log_annot,    'cmdb_id', 'log_name', 'log_name')

# Combine runs and filter length >= 2
all_runs = pd.concat([metric_runs, trace_runs, log_runs], ignore_index=True)
all_runs = all_runs[all_runs['run_length_in_minutes'] >= 2].copy()

# Sort by relative_deviation descending and select top 20
all_runs = all_runs.sort_values('relative_deviation', ascending=False).reset_index(drop=True)
top20_runs = all_runs.head(20).copy()
# For readability, round numeric columns
top20_runs['run_max_value'] = top20_runs['run_max_value'].round(6)
top20_runs['group_p95'] = top20_runs['group_p95'].round(6)
top20_runs['relative_deviation'] = top20_runs['relative_deviation'].round(6)
# rename run_length for output as requested
top20_runs = top20_runs.rename(columns={'run_length_in_minutes':'run_length'})

# Prepare raw anomaly rows for top 3 runs (limit 20 rows per run)
top3 = top20_runs.head(3).copy()
raw_blocks = []
for _, r in top3.iterrows():
    comp = r['component']
    kpi = r['KPI name']
    start = r['run_start_timestamp']
    end = r['run_end_timestamp']
    # determine which annotated DF to query by checking presence in metric_p95/trace_p95/log_p95
    # build candidate filters across metric_annot, trace_annot, log_annot
    # fetch anomalies matching comp & KPI within [start,end]
    block = pd.DataFrame()
    # try metric
    m = metric_annot[(metric_annot['is_anomaly']) & (metric_annot['cmdb_id']==comp) & (metric_annot['kpi_name']==kpi) & (metric_annot['dt']>=start) & (metric_annot['dt']<=end)]
    if not m.empty:
        block = m[['dt','cmdb_id','kpi_name','value']].rename(columns={'dt':'timestamp','kpi_name':'KPI name'})
    else:
        t = trace_annot[(trace_annot['is_anomaly']) & (trace_annot['cmdb_id']==comp) & (trace_annot['trace_name']==kpi) & (trace_annot['dt']>=start) & (trace_annot['dt']<=end)]
        if not t.empty:
            block = t[['dt','cmdb_id','trace_name','value']].rename(columns={'dt':'timestamp','trace_name':'KPI name'})
        else:
            l = log_annot[(log_annot['is_anomaly']) & (log_annot['cmdb_id']==comp) & (log_annot['log_name']==kpi) & (log_annot['dt']>=start) & (log_annot['dt']<=end)]
            if not l.empty:
                block = l[['dt','cmdb_id','log_name','value']].rename(columns={'dt':'timestamp','log_name':'KPI name'})
    if not block.empty:
        block = block.sort_values('timestamp').head(20)
        block['_group_key'] = f"{comp} | {kpi}"
        raw_blocks.append(block)

if raw_blocks:
    top3_raw_anomalies = pd.concat(raw_blocks, ignore_index=True)
else:
    top3_raw_anomalies = pd.DataFrame(columns=['timestamp','cmdb_id','KPI name','value','_group_key'])

# Final compact outputs:
# - top20_runs with required columns
# - top3_raw_anomalies (<= 20 rows per run, concatenated blocks)
top20_runs[['component','KPI name','run_start_timestamp','run_end_timestamp','run_length','run_max_value','group_p95','relative_deviation']].head(20), top3_raw_anomalies.head(60)

```
Out[3]:
```
```python
# Compose a compact plain-English summary of the run-detection results using existing kernel variables.
# Reuse: top20_runs, top3_raw_anomalies, all_runs (if available)
import pandas as pd

# Fallback guards (should not trigger in this session)
try:
    top20_runs
except NameError:
    top20_runs = pd.DataFrame()

try:
    top3_raw_anomalies
except NameError:
    top3_raw_anomalies = pd.DataFrame()

try:
    all_runs
except NameError:
    all_runs = pd.DataFrame()

lines = []
lines.append("Anomaly run detection summary (window: 2021-03-10 15:30–16:00 UTC+8).")
lines.append(f"Total multi-minute runs found: {len(all_runs)} (filtered to runs with length >=2).")
lines.append("Top runs (by relative deviation, up to 20):")

if top20_runs.empty:
    lines.append("  - No multi-minute anomaly runs detected.")
else:
    # show top 5 in-line for compactness
    for i, r in top20_runs.head(5).iterrows():
        comp = r['component']
        kpi = r['KPI name']
        start = pd.to_datetime(r['run_start_timestamp']).isoformat()
        end = pd.to_datetime(r['run_end_timestamp']).isoformat()
        length = int(r['run_length'])
        vmax = float(r['run_max_value'])
        p95 = float(r['group_p95']) if pd.notna(r['group_p95']) else float('nan')
        rel = float(r['relative_deviation'])
        lines.append(f"  {i+1}. {comp} | {kpi} — {start} to {end} ({length} min); max={vmax:g}, p95={p95:g}, rel_dev={rel:.3f}")

    lines.append("... (remaining top runs available in 'top20_runs' variable)")

# Top 3 raw anomaly blocks info
if top3_raw_anomalies.empty:
    lines.append("No raw anomaly rows available for top runs.")
else:
    # Identify group blocks included
    groups = top3_raw_anomalies['_group_key'].unique().tolist() if '_group_key' in top3_raw_anomalies.columns else []
    lines.append(f"Raw anomaly rows provided for top {len(groups)} run(s):")
    for g in groups:
        cnt = top3_raw_anomalies[top3_raw_anomalies['_group_key']==g].shape[0]
        lines.append(f"  - {g} : {cnt} rows (up to 20)")

lines.append("")
lines.append("Key observations:")
lines.append("  - Highest relative deviation: IG02 OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT ran all 31 minutes (15:30–16:00) with run_max=1 and group_p95=0 (rel_dev reported as 1.0 by calculation fallback).")
lines.append("  - Strong secondary spikes (rel_dev ~0.69) occurred for trace.duration_p95 KPIs (Tomcat01 / MG02) over 2-minute runs.")
lines.append("  - Many runs are short (2 minutes) with modest deviations (0.09–0.32), affecting trace, CPU and log KPIs across Tomcat, MG, docker, and apache components.")
lines.append("")
lines.append("Detailed tables and raw rows are stored in variables: top20_runs (runs), top3_raw_anomalies (raw rows for top 3 runs), and all_runs (full runs).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       component                                         KPI name       run_start_timestamp         run_end_timestamp  run_length  run_max_value    group_p95  relative_deviation
0           IG02   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-10 15:30:00+08:00 2021-03-10 16:00:00+08:00          31       1.000000     0.000000            1.000000
1       Tomcat01                       trace.to_MG02.duration_p95 2021-03-10 15:36:00+08:00 2021-03-10 15:37:00+08:00           2      38.924400    23.057982            0.688110
2           MG02                 trace.from_Tomcat01.duration_p95 2021-03-10 15:36:00+08:00 2021-03-10 15:37:00+08:00           2      38.924400    23.057982            0.688110
3           MG02                             trace.self.row_count 2021-03-10 15:30:00+08:00 2021-03-10 15:31:00+08:00           2    2006.000000  1520.700000            0.319129
4       apache02                          OSLinux-CPU_CPU_CPULoad 2021-03-10 15:47:00+08:00 2021-03-10 15:48:00+08:00           2       1.360000     1.062500            0.280000
5       dockerB1                          trace.self.duration_p95 2021-03-10 15:36:00+08:00 2021-03-10 15:37:00+08:00           2      12.089900    10.063322            0.201383
6           MG02                trace.from_dockerA1.duration_mean 2021-03-10 15:37:00+08:00 2021-03-10 15:38:00+08:00           2      12.187214    10.274606            0.186149
7       dockerA1                      trace.to_MG02.duration_mean 2021-03-10 15:37:00+08:00 2021-03-10 15:38:00+08:00           2      12.187214    10.274606            0.186149
8        Redis01                       OSLinux-CPU_CPU_CPUSysTime 2021-03-10 15:30:00+08:00 2021-03-10 15:31:00+08:00           2       3.083664     2.633900            0.170760
9       dockerB1                     trace.from_MG02.duration_p95 2021-03-10 15:36:00+08:00 2021-03-10 15:37:00+08:00           2      47.687750    41.957662            0.136568
10          MG02                   trace.to_dockerB1.duration_p95 2021-03-10 15:36:00+08:00 2021-03-10 15:37:00+08:00           2      47.687750    41.957662            0.136568
11          IG02  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum 2021-03-10 15:30:00+08:00 2021-03-10 15:31:00+08:00           2    1652.000000  1454.000000            0.136176
12  ServiceTest9                                          app.mrt 2021-03-10 15:37:00+08:00 2021-03-10 15:38:00+08:00           2      11.859680    10.503440            0.129123
13      Tomcat04                                    log.row_count 2021-03-10 15:30:00+08:00 2021-03-10 15:31:00+08:00           2     585.000000   518.500000            0.128255
14      Tomcat04                             trace.self.row_count 2021-03-10 15:30:00+08:00 2021-03-10 15:31:00+08:00           2    3837.000000  3457.700000            0.109697
15  ServiceTest3                                          app.mrt 2021-03-10 15:37:00+08:00 2021-03-10 15:38:00+08:00           2      12.054090    10.968795            0.098944
16          IG02                       OSLinux-CPU_CPU_CPUCpuUtil 2021-03-10 15:33:00+08:00 2021-03-10 15:34:00+08:00           2      31.528200    28.826200            0.093734
17          MG01                trace.from_dockerA2.duration_mean 2021-03-10 15:37:00+08:00 2021-03-10 15:38:00+08:00           2      11.501812    10.561496            0.089032
18      dockerA2                      trace.to_MG01.duration_mean 2021-03-10 15:37:00+08:00 2021-03-10 15:38:00+08:00           2      11.501812    10.561496            0.089032
19      dockerB2                      trace.to_MG02.duration_mean 2021-03-10 15:37:00+08:00 2021-03-10 15:38:00+08:00           2      11.844081    10.900926            0.086521,                    timestamp   cmdb_id                                        KPI name     value                                         _group_key
0  2021-03-10 15:30:00+08:00      IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   0.00000  IG02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLO...
1  2021-03-10 15:31:00+08:00      IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   0.00000  IG02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLO...
2  2021-03-10 15:32:00+08:00      IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   0.00000  IG02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLO...
3  2021-03-10 15:33:00+08:00      IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   0.00000  IG02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLO...
4  2021-03-10 15:34:00+08:00      IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   1.00000  IG02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLO...
..                       ...       ...                                             ...       ...                                                ...
19 2021-03-10 15:49:00+08:00      IG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT   0.00000  IG02 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLO...
20 2021-03-10 15:36:00+08:00  Tomcat01                      trace.to_MG02.duration_p95  38.92440              Tomcat01 | trace.to_MG02.duration_p95
21 2021-03-10 15:37:00+08:00  Tomcat01                      trace.to_MG02.duration_p95  23.46575              Tomcat01 | trace.to_MG02.duration_p95
22 2021-03-10 15:36:00+08:00      MG02                trace.from_Tomcat01.duration_p95  38.92440            MG02 | trace.from_Tomcat01.duration_p95
23 2021-03-10 15:37:00+08:00      MG02                trace.from_Tomcat01.duration_p95  23.46575            MG02 | trace.from_Tomcat01.duration_p95

[24 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_df and metric_p95 if available; otherwise load and compute global p95
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure dt (tz-aware) column
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure value numeric
metric_df['value'] = pd.to_numeric(metric_df.get('value', pd.Series()), errors='coerce')

# Ensure metric_p95 exists (global P95 per (cmdb_id,kpi_name)) computed on full series BEFORE filtering
try:
    metric_p95
except NameError:
    metric_p95 = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(
        count='count', p95_value=lambda x: x.quantile(0.95)
    ).reset_index().rename(columns={'cmdb_id':'cmdb_id','kpi_name':'kpi_name'})

# Normalize metric_p95 column names if needed
# After grouping, metric_p95 should have columns ['cmdb_id','kpi_name','count','p95_value']
# If different names exist (e.g., 'component'/'KPI name'), adapt
if 'component' in metric_p95.columns and 'KPI name' in metric_p95.columns:
    metric_p95 = metric_p95.rename(columns={'component':'cmdb_id','KPI name':'kpi_name'})

# Define time window (UTC+8)
start_dt = tz.localize(pd.Timestamp('2021-03-10 15:30:00'))
end_dt   = tz.localize(pd.Timestamp('2021-03-10 16:00:00'))

# 1) Filter metric rows for MG02 within the window
mg02_win = metric_df[(metric_df['cmdb_id']=='MG02') & (metric_df['dt']>=start_dt) & (metric_df['dt']<=end_dt)].copy()

# 1) Mark anomalies where value >= group's global P95 (merge p95 from metric_p95 which is global)
mg02_win = mg02_win.merge(metric_p95[['cmdb_id','kpi_name','p95_value']], on=['cmdb_id','kpi_name'], how='left')
mg02_win['is_anomaly'] = mg02_win['value'].notna() & mg02_win['p95_value'].notna() & (mg02_win['value'] >= mg02_win['p95_value'])

# 2) For each MG02 KPI compute required aggregations among anomaly rows
mg02_anoms = mg02_win[mg02_win['is_anomaly']].copy()

# Aggregate per KPI
if not mg02_anoms.empty:
    mg02_kpi_summary = mg02_anoms.groupby('kpi_name').agg(
        anomaly_count=('is_anomaly','sum'),
        earliest_anomaly_timestamp=('dt','min'),
        group_p95=('p95_value','first'),  # same per group
        run_max_value=('value','max')
    ).reset_index()
    # relative_deviation calculation
    def rel_dev(row):
        gp = row['group_p95']
        rm = row['run_max_value']
        if pd.isna(gp) or gp == 0:
            return rm
        else:
            return (rm - gp) / gp
    mg02_kpi_summary['relative_deviation'] = mg02_kpi_summary.apply(rel_dev, axis=1)
    # Sort by relative_deviation desc and keep top 20
    mg02_kpi_summary = mg02_kpi_summary.sort_values('relative_deviation', ascending=False).reset_index(drop=True)
    # Format numeric columns for compactness
    mg02_kpi_summary['group_p95'] = mg02_kpi_summary['group_p95'].round(6)
    mg02_kpi_summary['run_max_value'] = mg02_kpi_summary['run_max_value'].round(6)
    mg02_kpi_summary['relative_deviation'] = mg02_kpi_summary['relative_deviation'].round(6)
    mg02_summary_top20 = mg02_kpi_summary[['kpi_name','anomaly_count','earliest_anomaly_timestamp','group_p95','run_max_value','relative_deviation']].head(20)
else:
    mg02_summary_top20 = pd.DataFrame(columns=['kpi_name','anomaly_count','earliest_anomaly_timestamp','group_p95','run_max_value','relative_deviation'])

# 4) For top 5 KPIs, return up to 20 raw metric anomaly rows each with columns [timestamp, cmdb_id, kpi_name, value]
top5_kpis = mg02_summary_top20['kpi_name'].head(5).tolist()
raw_blocks = []
for kpi in top5_kpis:
    block = mg02_anoms[mg02_anoms['kpi_name']==kpi].sort_values('dt').head(20)
    if not block.empty:
        block_out = block[['dt','cmdb_id','kpi_name','value']].rename(columns={'dt':'timestamp'})
        block_out['_kpi_group'] = kpi
        raw_blocks.append(block_out)
if raw_blocks:
    mg02_top5_raw_anomalies = pd.concat(raw_blocks, ignore_index=True)
else:
    mg02_top5_raw_anomalies = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','_kpi_group'])

# Final compact outputs (variables to display)
mg02_summary_top20, mg02_top5_raw_anomalies

```
Out[4]:
```
```python
# Compose a concise plain-English summary for component MG02 using existing results:
# Variables available from prior steps: mg02_summary_top20, mg02_top5_raw_anomalies

import pandas as pd

# Fallback guards
try:
    df = mg02_summary_top20
    raw = mg02_top5_raw_anomalies
except NameError:
    df = pd.DataFrame()
    raw = pd.DataFrame()

lines = []
lines.append("MG02 metric anomaly summary within 2021-03-10 15:30–16:00 (UTC+8):")

if df.empty:
    lines.append("  - No metric anomalies detected for MG02 in the window.")
else:
    total_kpis = df.shape[0]
    total_anoms = int(df['anomaly_count'].sum())
    lines.append(f"  - {total_kpis} KPI(s) showed anomalies (total anomaly rows = {total_anoms}).")
    lines.append("  - Top KPIs by relative deviation (top 5 shown):")
    for i, r in df.head(5).iterrows():
        kpi = r['kpi_name']
        cnt = int(r['anomaly_count'])
        t = pd.to_datetime(r['earliest_anomaly_timestamp']).isoformat() if pd.notna(r['earliest_anomaly_timestamp']) else "N/A"
        p95 = float(r['group_p95']) if pd.notna(r['group_p95']) else float('nan')
        vmax = float(r['run_max_value']) if pd.notna(r['run_max_value']) else float('nan')
        rel = float(r['relative_deviation']) if pd.notna(r['relative_deviation']) else float('nan')
        lines.append(f"    {i+1}. {kpi} — anomalies={cnt}, earliest={t}, p95={p95:g}, max={vmax:g}, rel_dev={rel:.3f}")
    lines.append("  - Note: Most KPI anomaly_counts = 1 (single-point spikes).")

    if raw.empty:
        lines.append("  - No raw anomaly rows available for top KPIs.")
    else:
        groups = raw['_kpi_group'].unique().tolist() if '_kpi_group' in raw.columns else []
        lines.append(f"  - Raw anomaly rows provided for top {len(groups)} KPI(s) (up to 20 rows each).")
        # show one example timestamp per group
        for g in groups[:5]:
            sample_times = raw[raw['_kpi_group']==g]['timestamp'].head(3).dt.isoformat().tolist() if not raw[raw['_kpi_group']==g].empty else []
            sample_times_str = ", ".join(sample_times) if sample_times else "no rows"
            lines.append(f"    * {g}: sample timestamps: {sample_times_str}")

lines.append("")
lines.append("Key observations:")
lines.append("  - The largest relative deviation is for OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT (rel_dev ~9.0) with max=4 vs p95=0.4 — a pronounced single-point spike.")
lines.append("  - Other notable single-point spikes include CPU, disk and network KPIs with relative deviations from ~0.12 to ~0.82.")
lines.append("  - These are mostly isolated spikes (anomaly_count=1); investigate those timestamps and corresponding logs/traces if needed.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  anomaly_count earliest_anomaly_timestamp      group_p95  run_max_value  relative_deviation
0        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              1  2021-03-10 15:37:00+08:00       0.400000         4.0000            9.000000
1      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              1  2021-03-10 15:45:00+08:00       0.550000         1.0000            0.818182
2                              OSLinux-CPU_CPU_CPUWio              1  2021-03-10 15:50:00+08:00       0.406810         0.7072            0.738404
3                             OSLinux-CPU_CPU_CPULoad              1  2021-03-10 15:36:00+08:00       1.596000         2.1400            0.340852
4   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...              1  2021-03-10 15:40:00+08:00      14.250000        17.0000            0.192982
5                     OSLinux-CPU_CPU-0_SingleCpuUtil              1  2021-03-10 15:39:00+08:00      87.296770       100.0000            0.145518
6    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite              1  2021-03-10 15:39:00+08:00      15.306700        17.4667            0.141115
7                          OSLinux-CPU_CPU_CPUSysTime              1  2021-03-10 15:30:00+08:00       1.305030         1.4820            0.135606
8   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1  2021-03-10 15:47:00+08:00       0.355000         0.4000            0.126761
9     OSLinux-OSLinux_NETWORK_ens160_NETBandwidthUtil              1  2021-03-10 15:51:00+08:00       0.007855         0.0088            0.120306
10        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn              1  2021-03-10 15:31:00+08:00    1542.250000      1657.0000            0.074404
11       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut              1  2021-03-10 15:31:00+08:00    1453.000000      1561.0000            0.074329
12          JVM-Operating System_7779_JVM_JVM_CPULoad              1  2021-03-10 15:30:00+08:00       2.849800         3.0569            0.072672
13    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec              1  2021-03-10 15:51:00+08:00  599136.550000    638188.0000            0.065180
14         JVM-Memory_7779_JVM_Memory_HeapMemoryUsage              1  2021-03-10 15:42:00+08:00      54.066180        56.2280            0.039985
15                    OSLinux-CPU_CPU-1_SingleCpuUtil              1  2021-03-10 15:51:00+08:00       6.845380         7.0667            0.032331
16    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps              1  2021-03-10 15:39:00+08:00       1.036685         1.0667            0.028953
17                         OSLinux-CPU_CPU_CPUCpuUtil              1  2021-03-10 15:30:00+08:00      28.770245        29.4855            0.024861
18                    OSLinux-CPU_CPU-2_SingleCpuUtil              1  2021-03-10 15:57:00+08:00       5.506960         5.6368            0.023577
19    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum              1  2021-03-10 15:41:00+08:00    5093.700000      5194.0000            0.019691,                   timestamp cmdb_id                                           kpi_name    value                                         _kpi_group
0 2021-03-10 15:37:00+08:00    MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT   4.0000       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
1 2021-03-10 15:45:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps   1.0000     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
2 2021-03-10 15:50:00+08:00    MG02                             OSLinux-CPU_CPU_CPUWio   0.7072                             OSLinux-CPU_CPU_CPUWio
3 2021-03-10 15:36:00+08:00    MG02                            OSLinux-CPU_CPU_CPULoad   2.1400                            OSLinux-CPU_CPU_CPULoad
4 2021-03-10 15:40:00+08:00    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  17.0000  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames if present; otherwise load
try:
    metric_df
    trace_df
    log_df
    error_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')
    error_df = pd.read_csv('error_logs.csv')

# Ensure dt tz-aware column exists (UTC -> Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_df):
    if 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Metric raw rows for MG02 for selected KPIs within 15:34–15:38 UTC+8, up to 20 rows per KPI
selected_kpis = [
    "OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps",
    "OSLinux-CPU_CPU_CPUWio",
    "OSLinux-CPU_CPU_CPULoad",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead"
]

m_start = tz.localize(pd.Timestamp('2021-03-10 15:34:00'))
m_end   = tz.localize(pd.Timestamp('2021-03-10 15:38:00'))

mg02_metric_window = metric_df[
    (metric_df['cmdb_id']=='MG02') &
    (metric_df['kpi_name'].isin(selected_kpis)) &
    (metric_df['dt'] >= m_start) & (metric_df['dt'] <= m_end)
].copy()

# Sort and limit 20 rows per KPI, add group label column
mg02_metric_grouped_parts = []
for kpi, grp in mg02_metric_window.sort_values(['kpi_name','dt']).groupby('kpi_name'):
    part = grp[['dt','cmdb_id','kpi_name','value']].head(20).copy()
    part['_kpi_group'] = kpi
    mg02_metric_grouped_parts.append(part)
if mg02_metric_grouped_parts:
    mg02_metric_grouped = pd.concat(mg02_metric_grouped_parts, ignore_index=True)
else:
    mg02_metric_grouped = pd.DataFrame(columns=['dt','cmdb_id','kpi_name','value','_kpi_group'])

# 2) Trace rows related to Tomcat01 ↔ MG02 path for cmdb_id in ['Tomcat01','MG02'], time 15:35–15:37 UTC+8 (up to 50 rows)
t_start = tz.localize(pd.Timestamp('2021-03-10 15:35:00'))
t_end   = tz.localize(pd.Timestamp('2021-03-10 15:37:00'))

mask_name = (
    trace_df['trace_name'].str.contains('to_MG02', na=False) |
    trace_df['trace_name'].str.contains('from_Tomcat01', na=False) |
    (trace_df['trace_name'].str.contains('Tomcat01', na=False) & trace_df['trace_name'].str.contains('MG02', na=False))
)

trace_mg_tomcat = trace_df[
    (trace_df['cmdb_id'].isin(['Tomcat01','MG02'])) &
    (trace_df['dt'] >= t_start) & (trace_df['dt'] <= t_end) &
    mask_name
].copy()

trace_mg_tomcat_top50 = trace_mg_tomcat[['dt','cmdb_id','trace_name','value']].sort_values(['trace_name','dt']).head(50)

# 3) Logs and error_logs for cmdb_id='MG02' within 15:30–16:00 UTC+8 (limit 20 rows each)
win_start = tz.localize(pd.Timestamp('2021-03-10 15:30:00'))
win_end   = tz.localize(pd.Timestamp('2021-03-10 16:00:00'))

mg02_log_rows = log_df[
    (log_df['cmdb_id']=='MG02') &
    (log_df['dt'] >= win_start) & (log_df['dt'] <= win_end)
].copy()
mg02_log_rows_top20 = mg02_log_rows[['dt','cmdb_id','log_name','value']].sort_values('dt').head(20)

mg02_error_rows = error_df[
    (error_df['cmdb_id']=='MG02') &
    (error_df['dt'] >= win_start) & (error_df['dt'] <= win_end)
].copy()
mg02_error_rows_top20 = mg02_error_rows[['dt','cmdb_id','message']].sort_values('dt').head(20)

# Compact outputs (variables to display)
# Label blocks by variable names:
# - mg02_metric_grouped : raw metric rows per KPI (column _kpi_group labels the group)
# - trace_mg_tomcat_top50 : trace rows related to Tomcat01 <-> MG02 (up to 50)
# - mg02_log_rows_top20 : log rows for MG02 in window (up to 20)
# - mg02_error_rows_top20 : error_log rows for MG02 in window (up to 20)

mg02_metric_grouped, trace_mg_tomcat_top50, mg02_log_rows_top20, mg02_error_rows_top20

```
Out[5]:
```
```python
# Compact plain-English summary using existing variables from the session:
# mg02_metric_grouped, trace_mg_tomcat_top50, mg02_log_rows_top20, mg02_error_rows_top20

import pandas as pd

# Fallback guards
try:
    mm = mg02_metric_grouped
except NameError:
    mm = pd.DataFrame()

try:
    tr = trace_mg_tomcat_top50
except NameError:
    tr = pd.DataFrame()

try:
    lg = mg02_log_rows_top20
except NameError:
    lg = pd.DataFrame()

try:
    er = mg02_error_rows_top20
except NameError:
    er = pd.DataFrame()

lines = []
lines.append("Focused telemetry confirmation for MG02 (selected KPIs and Tomcat01↔MG02 traces).")
lines.append("")
# Metrics
lines.append("1) Metric raw rows (MG02) for selected KPIs within 2021-03-10 15:34–15:38 UTC+8:")
if mm.empty:
    lines.append("   - No matching metric rows found in that time range.")
else:
    groups = mm['_kpi_group'].unique().tolist()
    for g in groups:
        grp = mm[mm['_kpi_group']==g].sort_values('dt')
        lines.append(f"   - KPI: {g} — {len(grp)} row(s).")
        # show notable values for the KPI: max and timestamp(s)
        vmax = grp['value'].max()
        vt = grp[grp['value']==vmax]['dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z').tolist()
        lines.append(f"     sample max={vmax:g} at {', '.join(vt)}")
    # Highlight specific notable findings if present
    if 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT' in groups:
        tcp = mm[mm['_kpi_group']=='OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT']
        if not tcp.empty:
            t_spike = tcp[tcp['value']>0]
            if not t_spike.empty:
                lines.append(f"     -> Notable: TCP-FIN-WAIT spike value={int(t_spike.iloc[0]['value'])} at {t_spike.iloc[0]['dt'].strftime('%Y-%m-%d %H:%M:%S%z')}.")
    if 'OSLinux-CPU_CPU_CPULoad' in groups:
        cpu = mm[mm['_kpi_group']=='OSLinux-CPU_CPU_CPULoad']
        if not cpu.empty:
            lines.append(f"     -> CPU load sample: {cpu[['dt','value']].to_string(index=False, header=False)}")
    if 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead' in groups or 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps' in groups:
        lines.append("     -> Disk metrics in this window show small or zero values (no sustained high disk I/O).")

lines.append("")
# Traces
lines.append("2) Trace rows for Tomcat01 <-> MG02 within 2021-03-10 15:35–15:37 UTC+8:")
if tr.empty:
    lines.append("   - No matching trace rows found.")
else:
    lines.append(f"   - {len(tr)} rows returned (sample up to limit). Key observations:")
    # check for duration_p95 spikes at 15:36
    mask_p95 = tr['trace_name'].str.contains('duration_p95', na=False)
    if mask_p95.any():
        p95_rows = tr[mask_p95].sort_values(['cmdb_id','dt'])
        for _, r in p95_rows.iterrows():
            lines.append(f"     * {r['cmdb_id']} {r['trace_name']} = {r['value']:.6g} at {r['dt'].strftime('%Y-%m-%d %H:%M:%S%z')}")
    # row_count increase
    mask_rowc = tr['trace_name'].str.contains('row_count', na=False)
    if mask_rowc.any():
        rc = tr[mask_rowc].sort_values('dt')
        for _, r in rc.iterrows():
            lines.append(f"     * {r['cmdb_id']} {r['trace_name']} = {int(r['value'])} at {r['dt'].strftime('%Y-%m-%d %H:%M:%S%z')}")
    lines.append("   - Notable: duration_p95 on Tomcat01 and MG02 spikes at 15:36 to ~38.92, and row_count rises to 256 at 15:37 (indicating increased traffic/latency on Tomcat01↔MG02).")

lines.append("")
# Logs and error_logs
lines.append("3) Logs / error_logs for MG02 during 2021-03-10 15:30–16:00 UTC+8:")
if lg.empty and er.empty:
    lines.append("   - No log.csv or error_logs.csv rows found for MG02 in the incident window.")
else:
    if not lg.empty:
        lines.append(f"   - log.csv: {len(lg)} row(s) (sample shown in variable).")
    else:
        lines.append("   - log.csv: none found.")
    if not er.empty:
        lines.append(f"   - error_logs.csv: {len(er)} row(s).")
    else:
        lines.append("   - error_logs.csv: none found.")

lines.append("")
lines.append("Summary conclusion:")
lines.append("  - Metric: A clear TCP-FIN-WAIT single-point spike at 15:37 (value=4) on MG02; short CPU load spike at 15:36 (value=2.14); disk metrics mostly low/zero in this short window.")
lines.append("  - Trace: Tomcat01 ↔ MG02 shows synchronized spikes at 15:36 (duration_p95 ~38.9) and increased row_count at 15:37 (256), supporting a brief latency/traffic event between Tomcat01 and MG02.")
lines.append("  - Logs: No MG02 application logs or error log entries captured in the window (no direct log evidence).")
lines.append("")
lines.append("Raw blocks are stored in variables: mg02_metric_grouped (grouped per KPI), trace_mg_tomcat_top50, mg02_log_rows_top20, mg02_error_rows_top20.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                          dt cmdb_id                                         kpi_name   value                                       _kpi_group
0  2021-03-10 15:35:00+08:00    MG02                          OSLinux-CPU_CPU_CPULoad  1.3300                          OSLinux-CPU_CPU_CPULoad
1  2021-03-10 15:36:00+08:00    MG02                          OSLinux-CPU_CPU_CPULoad  2.1400                          OSLinux-CPU_CPU_CPULoad
2  2021-03-10 15:38:00+08:00    MG02                          OSLinux-CPU_CPU_CPULoad  1.2200                          OSLinux-CPU_CPU_CPULoad
3  2021-03-10 15:35:00+08:00    MG02                           OSLinux-CPU_CPU_CPUWio  0.0208                           OSLinux-CPU_CPU_CPUWio
4  2021-03-10 15:36:00+08:00    MG02                           OSLinux-CPU_CPU_CPUWio  0.0291                           OSLinux-CPU_CPU_CPUWio
5  2021-03-10 15:38:00+08:00    MG02                           OSLinux-CPU_CPU_CPUWio  0.0166                           OSLinux-CPU_CPU_CPUWio
6  2021-03-10 15:35:00+08:00    MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  0.0000   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps
7  2021-03-10 15:35:00+08:00    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  0.0000  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
8  2021-03-10 15:34:00+08:00    MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.0000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
9  2021-03-10 15:35:00+08:00    MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.0000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
10 2021-03-10 15:37:00+08:00    MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  4.0000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
11 2021-03-10 15:38:00+08:00    MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.0000     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT,                             dt   cmdb_id                         trace_name       value
1263 2021-03-10 15:35:00+08:00      MG02  trace.from_Tomcat01.duration_mean    3.162769
1491 2021-03-10 15:36:00+08:00      MG02  trace.from_Tomcat01.duration_mean    6.278116
1719 2021-03-10 15:37:00+08:00      MG02  trace.from_Tomcat01.duration_mean    4.163660
1264 2021-03-10 15:35:00+08:00      MG02   trace.from_Tomcat01.duration_p95   18.340800
1492 2021-03-10 15:36:00+08:00      MG02   trace.from_Tomcat01.duration_p95   38.924400
1720 2021-03-10 15:37:00+08:00      MG02   trace.from_Tomcat01.duration_p95   23.465750
1265 2021-03-10 15:35:00+08:00      MG02      trace.from_Tomcat01.row_count  225.000000
1493 2021-03-10 15:36:00+08:00      MG02      trace.from_Tomcat01.row_count  225.000000
1721 2021-03-10 15:37:00+08:00      MG02      trace.from_Tomcat01.row_count  256.000000
1314 2021-03-10 15:35:00+08:00  Tomcat01        trace.to_MG02.duration_mean    3.162769
1542 2021-03-10 15:36:00+08:00  Tomcat01        trace.to_MG02.duration_mean    6.278116
1770 2021-03-10 15:37:00+08:00  Tomcat01        trace.to_MG02.duration_mean    4.163660
1315 2021-03-10 15:35:00+08:00  Tomcat01         trace.to_MG02.duration_p95   18.340800
1543 2021-03-10 15:36:00+08:00  Tomcat01         trace.to_MG02.duration_p95   38.924400
1771 2021-03-10 15:37:00+08:00  Tomcat01         trace.to_MG02.duration_p95   23.465750
1316 2021-03-10 15:35:00+08:00  Tomcat01            trace.to_MG02.row_count  225.000000
1544 2021-03-10 15:36:00+08:00  Tomcat01            trace.to_MG02.row_count  225.000000
1772 2021-03-10 15:37:00+08:00  Tomcat01            trace.to_MG02.row_count  256.000000, Empty DataFrame
Columns: [dt, cmdb_id, log_name, value]
Index: [], Empty DataFrame
Columns: [dt, cmdb_id, message]
Index: [])```
```